# Local Nonlinear Support Assembly Demo

This notebook demonstrates nonlinear local support assembly with the same Python-side partition and support-patch workflow as the linear demo. Python handles METIS partitioning, support-patch construction, DoF/index bookkeeping, visualization, and verification. C++ receives the patch metadata and assembles core-sized nonlinear residuals and Jacobians over the supplied support elements.


In [ ]:
import math
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

from partition.metis import metis_partition_from_fes
from utils.patches import dofs_of_elements

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)


## Mesh, Space, And Nonlinear Form

The nonlinear form is a generic NGSolve `BilinearForm`. The C++ nonlinear assembler does not own the PDE model; it uses the supplied form and current global vector to compute element nonlinear residual and Jacobian contributions.

Here we use the same nonlinear volume operator as `demo_nonlinear_local_assembly.py`:

`F(u; v) = int grad(u) . grad(v) dx + int u^3 v dx`.

The demo compares the local nonlinear operator against the global nonlinear residual/Jacobian restricted to the chosen local DoFs, with the same Dirichlet row treatment on physical boundary DoFs.


In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.15))
fes = H1(mesh, order=1, dirichlet="left|right|top|bottom")
u, v = fes.TnT()

a = BilinearForm(fes)
a += (grad(u) * grad(v) + u * u * u * v) * dx

gfu = GridFunction(fes, name="u_current")
gfu.Set((x * (1 - x)) ** 2 + 0.25 * (y * (1 - y)) ** 2)

print("ne =", mesh.ne, "ndof =", fes.ndof)
Draw(gfu, mesh, "u_current")

free_dofs = fes.FreeDofs()
boundary_dofs = [i for i in range(fes.ndof) if not free_dofs[i]]
boundary_values = [0.0] * len(boundary_dofs)

print("Dirichlet boundary dofs:", len(boundary_dofs))


## METIS Element Partition And DoF Lists

`partition/metis.py` still partitions an element graph. For the new nonlinear DoF-only C++ API, Python converts those element lists into DoF lists only:

- `nonoverlapping_partition`: DoFs on the METIS core elements, used for bookkeeping and visualization.
- `overlapping_partition`: DoFs on the graph-expanded element partition, used as local unknowns.

C++ receives only `local_dofs = overlapping_partition[i]` and computes support elements/support DoFs internally with `BuildLocalSupportInfo`.


In [ ]:
core_element_partition, overlapping_element_partition, cutcount = metis_partition_from_fes(
    fes,
    nparts=9,
    overlap_width=1,
    free_dofs_only=False,
)

nonoverlapping_partition = [
    dofs_of_elements(fes, elements)
    for elements in core_element_partition
]
overlapping_partition = [
    dofs_of_elements(fes, elements)
    for elements in overlapping_element_partition
]
support_infos = [
    myassembling.BuildLocalSupportInfo(fes, local_dofs)
    for local_dofs in overlapping_partition
]

print("METIS cutcount =", cutcount)
print("core element counts =", [len(p) for p in core_element_partition])
print("overlapping element counts =", [len(p) for p in overlapping_element_partition])
print("nonoverlapping dof counts =", [len(p) for p in nonoverlapping_partition])
print("overlapping/local dof counts =", [len(p) for p in overlapping_partition])
print("support dof counts =", [len(info.support_dofs) for info in support_infos])
print("support element counts =", [len(info.support_elements) for info in support_infos])

for nonoverlap, overlap, info in zip(nonoverlapping_partition, overlapping_partition, support_infos):
    assert set(nonoverlap).issubset(set(overlap))
    assert list(info.core_dofs) == list(overlap)
    assert set(overlap).issubset(set(info.support_dofs))
    for k, dof in enumerate(info.core_dofs):
        assert info.support_dofs[info.core_in_support[k]] == dof


## Visualize The METIS Element Partition

Each plot shows one METIS core element partition. This is only the source used to build DoF lists; elements are not passed to the nonlinear C++ operator.


In [ ]:
l2 = L2(mesh, order=0)

for i, elements in enumerate(core_element_partition):
    omega_i = GridFunction(l2, name=f"Omega_{i}")
    omega_i.vec[:] = 0
    for elnr in elements:
        omega_i.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1.0
    Draw(omega_i, mesh, f"subdomain {i}: core elements=1")


## Inspect C++ Support Info

The support information below is computed by C++ from the overlapping DoFs. `overlap-extra dofs` are `overlapping_dofs - nonoverlapping_dofs`; `support-extra dofs` are `support_dofs - overlapping_dofs`.


In [ ]:
for i, (nonoverlap, overlap, info) in enumerate(zip(nonoverlapping_partition, overlapping_partition, support_infos)):
    nonoverlap_set = set(nonoverlap)
    overlap_set = set(overlap)
    support_set = set(info.support_dofs)

    print(f"Omega_{i}:")
    print("  nonoverlapping dofs:", len(nonoverlap_set))
    print("  overlapping/local dofs:", len(overlap_set))
    print("  overlap extra dofs:", len(overlap_set - nonoverlap_set))
    print("  support dofs:", len(support_set))
    print("  support extra dofs:", len(support_set - overlap_set))
    print("  support elements:", len(info.support_elements))


## Visualize DoF Layers And Support Elements

Each subplot shows one local problem. The data all come from the same C++ support construction used by the nonlinear operator:

- filled cells: `info.support_elements`, the integration domain
- blue points: non-overlapping DoFs
- red points: overlap-extra DoFs, `overlapping_dofs - nonoverlapping_dofs`
- purple points: support-extra DoFs, `support_dofs - overlapping_dofs`


In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon

volume_elements = list(mesh.Elements(VOL))

def vertex_xy(vnr):
    p = mesh.vertices[int(vnr)].point
    return float(p[0]), float(p[1])

def element_vertices(elnr):
    return [int(v.nr) for v in volume_elements[int(elnr)].vertices]

def element_polygon(elnr):
    return [vertex_xy(vnr) for vnr in element_vertices(elnr)]

def element_centroid(elnr):
    pts = element_polygon(elnr)
    return sum(p[0] for p in pts) / len(pts), sum(p[1] for p in pts) / len(pts)

def build_dof_coordinates():
    coords = {int(vnr): vertex_xy(vnr) for vnr in range(mesh.nv)}
    sums = {}
    counts = {}
    for el in volume_elements:
        elnr = int(el.nr)
        cx, cy = element_centroid(elnr)
        for dof in fes.GetDofNrs(ElementId(VOL, elnr)):
            dof = int(dof)
            if dof < 0 or dof in coords:
                continue
            sx, sy = sums.get(dof, (0.0, 0.0))
            sums[dof] = (sx + cx, sy + cy)
            counts[dof] = counts.get(dof, 0) + 1
    for dof, (sx, sy) in sums.items():
        coords[dof] = (sx / counts[dof], sy / counts[dof])
    return coords

dof_xy = build_dof_coordinates()

def draw_mesh_edges(ax):
    for el in volume_elements:
        pts = [vertex_xy(v.nr) for v in el.vertices]
        pts.append(pts[0])
        xs, ys = zip(*pts)
        ax.plot(xs, ys, color="0.82", linewidth=0.7, zorder=1)

def add_elements(ax, elements, color="tab:green", alpha=0.22):
    for elnr in elements:
        ax.add_patch(
            Polygon(
                element_polygon(elnr),
                closed=True,
                facecolor=color,
                edgecolor="0.45",
                linewidth=0.45,
                alpha=alpha,
                zorder=2,
            )
        )

def scatter_dofs(ax, dofs, color, label, size=42, zorder=4):
    pts = [dof_xy[d] for d in sorted(dofs) if d in dof_xy]
    if pts:
        xs, ys = zip(*pts)
        ax.scatter(xs, ys, s=size, color=color, edgecolor="black", linewidth=0.45, label=label, zorder=zorder)

all_vertex_pts = [vertex_xy(i) for i in range(mesh.nv)]
all_x, all_y = zip(*all_vertex_pts)
pad = 0.04
xmin, xmax = min(all_x) - pad, max(all_x) + pad
ymin, ymax = min(all_y) - pad, max(all_y) + pad

npatches = len(overlapping_partition)
cols = min(3, npatches)
rows = math.ceil(npatches / cols)
fig, axes = plt.subplots(rows, cols, figsize=(3.8 * cols, 3.8 * rows), squeeze=False)

for ax in axes.flat[npatches:]:
    ax.axis("off")

for patch_id, (nonoverlap_part, local_dofs, info) in enumerate(zip(nonoverlapping_partition, overlapping_partition, support_infos)):
    ax = axes.flat[patch_id]
    nonoverlap = set(nonoverlap_part)
    overlap = set(local_dofs)
    support_dofs = set(info.support_dofs)

    assert nonoverlap <= overlap
    assert overlap <= support_dofs
    assert list(info.core_dofs) == list(local_dofs)

    draw_mesh_edges(ax)
    add_elements(ax, info.support_elements)
    ax.scatter(all_x, all_y, s=8, color="0.72", zorder=3)
    scatter_dofs(ax, nonoverlap, "tab:blue", "non-overlap dofs", size=38, zorder=5)
    scatter_dofs(ax, overlap - nonoverlap, "tab:red", "overlap-extra dofs", size=42, zorder=6)
    scatter_dofs(ax, support_dofs - overlap, "tab:purple", "support-extra dofs", size=34, zorder=4)

    ax.set_title(f"Omega_{patch_id}")
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks([])
    ax.set_yticks([])

legend_handles = [
    Patch(facecolor="tab:green", edgecolor="0.45", alpha=0.22, label="support elements"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:blue", markeredgecolor="black", markersize=7, label="non-overlap dofs"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:red", markeredgecolor="black", markersize=7, label="overlap-extra dofs"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="tab:purple", markeredgecolor="black", markersize=7, label="support-extra dofs"),
]
fig.legend(handles=legend_handles, loc="upper center", ncol=4, frameon=True)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()


## Global Nonlinear Residual And Jacobian

For verification, assemble the raw global nonlinear residual and Jacobian using NGSolve's nonlinear form machinery:

- `a.Apply(gfu.vec, res_global)` evaluates the nonlinear residual at the current state.
- `a.AssembleLinearization(gfu.vec)` assembles the Jacobian at the same state, stored as `a.mat`.

The local C++ operator receives the global FEM Dirichlet DoFs and applies row-only treatment on those local rows: residual entry `u[d] - boundary_value`, and Jacobian row equal to the identity row. Boundary columns in non-boundary rows are not cleared. The comparison below applies the same row-only rule to the global reference entries.


In [ ]:
# global residual
res_global = gfu.vec.CreateVector()
a.Apply(gfu.vec, res_global)

# global jacobian matrix
a.AssembleLinearization(gfu.vec)
jac_global = a.mat


## Build Local Nonlinear Operators And Evaluate Them

For each subdomain, pass the overlapping DoFs and the global FEM Dirichlet data to C++:

```python
local_op = myassembling.LocalNonlinearOperator(
    fes, a, local_dofs, boundary_dofs, boundary_values
)
```

Here `boundary_values` are all zero. The boundary DoFs are physical Dirichlet DoFs from the global FESpace, not artificial local interface DoFs.


In [ ]:
local_operators = []
local_jacobians = []
local_residuals = []

for local_dofs in overlapping_partition:
    local_op = myassembling.LocalNonlinearOperator(
        fes, a, local_dofs, boundary_dofs, boundary_values
    )
    local_operators.append(local_op)
    local_jacobians.append(local_op.Jacobian(gfu.vec))
    local_residuals.append(local_op.Residual(gfu.vec))

for i, (local_dofs, info, jac_local, res_local) in enumerate(zip(overlapping_partition, support_infos, local_jacobians, local_residuals)):
    assert list(res_local.core_dofs) == list(local_dofs)
    assert list(jac_local.core_dofs) == list(local_dofs)
    assert list(res_local.support_dofs) == list(info.support_dofs)
    assert list(res_local.support_elements) == list(info.support_elements)
    print(f"Omega_{i}:")
    print("  nonoverlapping dofs:", len(nonoverlapping_partition[i]))
    print("  overlapping/local dofs:", len(local_dofs))
    print("  support dofs:", len(res_local.support_dofs))
    print("  support elements:", len(res_local.support_elements))
    print("  Jacobian shape:", (jac_local.mat.height, jac_local.mat.width))
    print("  Residual shape:", res_local.vec.size)


## Verify Against Global Nonlinear Assembly

The local nonlinear residual returned by C++ is the vector on `patch.core_dofs`, and the local nonlinear Jacobian is the local block on those same DoFs.

For non-Dirichlet rows, compare against the raw global restriction. For Dirichlet rows, compare against the same row-only treatment used locally:

`F_i = u[d_i] - boundary_value_i`

`J_i` row is the identity row.

Boundary columns in non-boundary rows are not cleared.

We do not compare full support objects with global support subblocks.


In [ ]:
def matrix_entry(mat, i, j):
    return float(mat[int(i), int(j)])

boundary_value = dict(zip(boundary_dofs, boundary_values))
boundary_set = set(boundary_dofs)

for patch_id, (local_dofs, res_local, jac_local) in enumerate(
    zip(overlapping_partition, local_residuals, local_jacobians)
):
    jac_err = 0.0
    for i, gi in enumerate(local_dofs):
        for j, gj in enumerate(local_dofs):
            if gi in boundary_set:
                reference = 1.0 if i == j else 0.0
            else:
                reference = matrix_entry(jac_global, gi, gj)
            jac_err = max(
                jac_err,
                abs(matrix_entry(jac_local.mat, i, j) - reference),
            )

    res_err = 0.0
    for i, d in enumerate(local_dofs):
        if d in boundary_set:
            reference = float(gfu.vec[d]) - boundary_value[d]
        else:
            reference = float(res_global[d])
        res_err = max(res_err, abs(float(res_local.vec[i]) - reference))

    print(f"Omega_{patch_id}:")
    print("  nonoverlapping dofs:", len(nonoverlapping_partition[patch_id]))
    print("  overlapping/local dofs:", len(local_dofs))
    print("  support dofs:", len(res_local.support_dofs))
    print("  support elements:", len(res_local.support_elements))
    print("  Jacobian inf error:", f"{jac_err:.3e}")
    print("  residual inf error:", f"{res_err:.3e}")

    if not math.isclose(jac_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear Jacobian does not match boundary-treated global local-dof restriction")
    if not math.isclose(res_err, 0.0, abs_tol=1e-10):
        raise RuntimeError("local nonlinear residual does not match boundary-treated global local-dof restriction")
